# Обучение Baseline++

Encoder–decoder с шириной 512, 8 головами, 5 encoder-блоками и 7 decoder-блоками.
В self-attention используется RoPE, в блоках — Pre-RMSNorm и SwiGLU.
Параметры берутся из `experiment_configs/baseline++.yaml`; текущий `dim_glu` равен 1024.

Ноутбук использует существующий `Experiment`: сохраняет конфиг, токенайзер,
индексы выборок, метрики и checkpoints. Обучение запускается отдельной ячейкой.
Генерация и pass@1 здесь пока не реализованы.

## 1. Импорты

Открой ноутбук из корня проекта или `notebooks/`.
После изменения Python-модулей перезапусти ядро.


In [ ]:
import sys
from pathlib import Path

project_dir = Path.cwd()
if not (project_dir / "lib").is_dir():
    project_dir = project_dir.parent
if not (project_dir / "lib").is_dir():
    raise FileNotFoundError("Открой ноутбук из корня HW-2 или папки notebooks/.")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

import matplotlib.pyplot as plt
import torch
from IPython.display import clear_output
from torch.nn import functional as F

from lib.train import train_loop
from lib.experiment import Experiment
from lib.utils.metrics import format_metrics, plot_losses, summarize_epoch
from lib.utils.experiment.checkpoint import capture_rng_state
from lib.utils.experiment.resume import restore_rng_state
from lib.utils.paths import PROJECT_DIR

from lib import BaselinePlusPlusModel


In [ ]:
CONFIG_PATH = PROJECT_DIR / "experiment_configs" / "baseline++.yaml"
DEVICE = "auto"
WEIGHTS_FROM = None  # Path to another run's best.pt for a fresh experiment.
RESUME_RUN = PROJECT_DIR / "experiment_runs" / "baseline_plus_plus_kaggle"
RESUME_MAX_TOKENS = None
RESUME_MINI_EPOCH = None


## Создание или восстановление эксперимента

Выбери один из трёх сценариев:

- **Новый запуск:** `Experiment.setup(CONFIG_PATH, device=DEVICE)`.
- **Новый запуск с весами:** тот же `setup`, но с `weights_from="experiment_runs/previous/checkpoints/best.pt"`.
- **Продолжение:** `Experiment.resume(RESUME_RUN, device=DEVICE)`.

Имя нового запуска задаётся в YAML; существующая папка вызывает ошибку.
При продолжении используются конфиг, токенайзер и выборки сохранённого запуска.
`DEVICE = "auto"` выбирает CUDA, MPS или CPU; устройство в конфиге не хранится.

`RESUME_MAX_TOKENS` и `RESUME_MINI_EPOCH` можно оставить равными `None`.
При смене бюджета токенов текущая полная эпоха начинается заново;
веса, оптимизатор, scheduler и история сохраняются. Часть примеров повторится.
При смене интервала мини-эпох следующий интервал считается от точки восстановления.
Scheduler сохраняет исходное расписание по числу шагов.


In [ ]:
if RESUME_RUN is None:
    experiment = Experiment.setup(
        CONFIG_PATH, weights_from=WEIGHTS_FROM, device=DEVICE,
        details={"notes": "Baseline++: Pre-RMSNorm, RoPE, SwiGLU"},
    )
else:
    experiment = Experiment.resume(
        RESUME_RUN, device=DEVICE,
        max_tokens=RESUME_MAX_TOKENS,
        examples_per_mini_epoch=RESUME_MINI_EPOCH,
    )

assert isinstance(experiment.model, BaselinePlusPlusModel)
print(f"Run: {experiment.run_dir}")
print(f"Device: {experiment.device}")
print(f"Completed epochs: {experiment.completed_epochs}")
print(f"Resume progress: {experiment.progress}")


## 3. Проверка собранной модели и данных

Проверяем число слоёв, размерность головы и передачу `max_seq_len` в RoPE.
Cross-attention использует выход энкодера и работает без RoPE.

In [ ]:
model = experiment.model
params = experiment.config["model"]["params"]
max_seq_len = params["max_seq_len"]
assert len(model.encoders) == params["num_encoder_layers"]
assert len(model.decoders) == params["num_decoder_layers"]
assert all(block.attention.rope.max_seq_len == max_seq_len for block in model.encoders)
assert all(block.self_attention.rope.max_seq_len == max_seq_len for block in model.decoders)
assert all(block.cross_attention.rope is None for block in model.decoders)

num_parameters = sum(parameter.numel() for parameter in model.parameters())
print(f"Model: {type(model).__name__}, parameters: {num_parameters:,}")
print(f"Encoder / decoder blocks: {len(model.encoders)} / {len(model.decoders)}")
print(f"Head dimension: {model.encoders[0].attention.head_dim}")
print(f"Vocabulary: {experiment.tokenizer.get_vocab_size():,}, pad_id: {experiment.pad_id}")
print(f"Train / validation examples: {len(experiment.train_loader.dataset):,} / {len(experiment.val_loader.dataset):,}")
print(f"Train / validation batches: {len(experiment.train_loader):,} / {len(experiment.val_loader):,}")
print(f"Optimizer: {type(experiment.optimizer).__name__}")
print(f"Learning rates: {[group['lr'] for group in experiment.optimizer.param_groups]}")
scheduler_name = type(experiment.scheduler).__name__ if experiment.scheduler is not None else "disabled"
print(f"Scheduler: {scheduler_name}")
mini_size = experiment.config["training"]["examples_per_mini_epoch"]
mini_epochs_per_epoch = (len(experiment.train_loader.dataset) + mini_size - 1) // mini_size
print(f"Mini-epoch: {mini_size:,} examples; per full epoch: {mini_epochs_per_epoch}")
print(f"Planned mini-epochs at current interval: {experiment.config['training']['num_epochs'] * mini_epochs_per_epoch}")
print("Resume preserves the original scheduler state and horizon.")
print(f"Scheduler config: {experiment.config['scheduler']}")


## 4. Один батч без обновления весов

Выполняй перед обучением или сразу после resume. Проверяем форму logits
и конечность loss на одном настоящем батче. Режим модели, генераторы случайных
чисел и эпоха загрузчика восстанавливаются в `finally`.

In [ ]:
rng_state = capture_rng_state(experiment.device)
model_was_training = experiment.model.training

try:
    experiment.train_loader.set_epoch(experiment.completed_epochs)
    batch = next(iter(experiment.train_loader))
    batch = {name: tensor.to(experiment.device) for name, tensor in batch.items()}
    experiment.model.eval()
    with torch.no_grad():
        logits = experiment.model(
            encoder_input_ids=batch["encoder_input_ids"],
            decoder_input_ids=batch["decoder_input_ids"],
            encoder_key_padding_mask=batch["encoder_key_padding_mask"],
            decoder_key_padding_mask=batch["decoder_key_padding_mask"],
        )
        vocab_size = experiment.tokenizer.get_vocab_size()
        assert logits.shape == (*batch["label_ids"].shape, vocab_size)
        batch_loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            batch["label_ids"].reshape(-1),
            ignore_index=experiment.pad_id,
        )
        assert torch.isfinite(batch_loss), "Batch loss is not finite."
    print(f"Logits shape: {tuple(logits.shape)}")
    print(f"Batch loss: {batch_loss.item():.4f}")
finally:
    experiment.model.train(model_was_training)
    experiment.train_loader.set_epoch(experiment.completed_epochs)
    restore_rng_state(rng_state, experiment.device)

del batch, logits, batch_loss

## Обучение и метрики

Ячейка ниже только определяет функции; обучение запускается отдельной ячейкой.
Цикл использует `experiment.progress`: при обычном resume уже обработанные
батчи пропускаются, при смене бюджета токенов эпоха начинается заново.

После мини-эпохи выполняются validation, шаг scheduler и один `save_checkpoint()`.
Он обновляет `last.pt`, при улучшении — `best.pt`, историю и краткую сводку.
Scheduler делает один шаг после каждой мини-эпохи внутри `train_loop`;
ноутбук отдельно его не вызывает.

Показываются эпоха, мини-эпоха, train/validation loss, время и секунды обучения
на пример. Время валидации, записи файлов и отрисовки в скорость обучения не входит.
График строит `lib.utils.metrics`; `show_metrics` можно заменить другим callback отображения.


In [ ]:
def show_metrics(experiment):
    clear_output(wait=True)
    print(format_metrics(experiment.validation_history[-1]))
    fig = plot_losses(experiment)
    plt.show()
    plt.close(fig)


def train_model(experiment):
    """The training loop stays here; Experiment only builds and saves state."""
    training = experiment.config["training"]
    mini_size = training["examples_per_mini_epoch"]

    def on_validation(metrics):
        experiment.record_validation({"epoch": epoch + 1, **metrics})
        epoch_finished = metrics["batch_index"] == len(experiment.train_loader)
        if epoch_finished:
            experiment.history.append(summarize_epoch(experiment.validation_history, epoch + 1))
        experiment.save_checkpoint()
        show_metrics(experiment)

    print(f"Completed epochs: {experiment.completed_epochs}/{training['num_epochs']}")
    for epoch in range(experiment.completed_epochs, training["num_epochs"]):
        experiment.train_loader.set_epoch(epoch)
        train_loop(
            model=experiment.model,
            train_loader=experiment.train_loader,
            optimizer=experiment.optimizer,
            device=experiment.device,
            pad_id=experiment.pad_id,
            val_loader=experiment.val_loader,
            examples_per_mini_epoch=mini_size,
            scheduler=experiment.scheduler,
            on_validation=on_validation,
            **experiment.progress,
        )


In [ ]:
train_model(experiment)

## Сводка

`results/summary.json` уже обновляется вместе с checkpoint. Эта необязательная
ячейка добавляет статус и сохраняет график; веса она не меняет.
После прерывания сначала сделай resume, чтобы сводка и график соответствовали
сохранённому состоянию, а не незавершённой работе в памяти.


In [ ]:
# Checkpoints already contain progress, and summary.json is updated on save.
experiment.save_results(details={
    "status": "completed" if experiment.completed_epochs >= experiment.config["training"]["num_epochs"] else "interrupted",
})

if experiment.history or experiment.validation_history:
    fig = plot_losses(experiment)
    fig.savefig(experiment.run_dir / "results" / "loss.png")
    plt.close(fig)

print(f"Completed epochs: {experiment.completed_epochs}")
print(f"Saved mini-epochs: {len(experiment.validation_history)}")
print(f"Best validation loss: {experiment.best_val_loss}")
print(f"Results: {experiment.run_dir / 'results'}")
